# YZM 513 — Week 6 Lab: Train One Neural Network, From NumPy to the GPU

**İstanbul Medeniyet Üniversitesi — AI Engineering M.Sc. (Fall 2026)**

We train **one** small neural network (an MLP that reads handwritten digits, MNIST) and carry it through the whole course so far:

| Step | What we do | Course week it reviews |
|---|---|---|
| **A** | Warm-up on the CPU: measure, vectorise, broadcast, BLAS threads | Weeks 1–4 |
| **B** | Write the MLP *as matrices*; check the gradients; train it with NumPy | Weeks 1, 2, 5 |
| **C** | Meet the GPU with simple array tasks (CuPy) | **Week 6** |
| **D** | Run the **same** training code on the GPU; find when the GPU wins | **Week 6** |
| **E** | Same model in PyTorch (autograd) and a mixed-precision preview | Weeks 9–12 preview |
| **F** | Summary table and your exercises | |

**How to run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all (about 5–8 minutes).
Cells marked 📝 ask you to write something down. Cells marked 🔧 are your turn.

> **Why this example?** An MLP on MNIST is small enough for a free T4 and for you to check by hand,
> yet *every* line of it is a matrix operation you already met in Weeks 1–5. Nothing is hidden inside
> a framework until Part E, so you can see exactly *what* the GPU is speeding up.

## 0. Setup

First check that Colab gave you a GPU. If `nvidia-smi` fails, switch the runtime type to **T4 GPU** and start again.

In [ ]:
!nvidia-smi

In [ ]:
# Install the CuPy build that matches Colab's CUDA driver (Colab usually has CuPy already).
import re, subprocess, sys
try:
    import cupy as cp
except ImportError:
    smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
    m = re.search(r"CUDA Version:\s*(\d+)", smi)
    assert m, "No NVIDIA GPU found. Runtime → Change runtime type → T4 GPU, then run again."
    pkg = f"cupy-cuda{m.group(1)}x"      # e.g. cupy-cuda12x
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
    import cupy as cp

In [ ]:
import os, time, math, urllib.request
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt

assert cp.cuda.runtime.getDeviceCount() > 0, "No GPU visible: switch the runtime to T4 GPU."

dev   = cp.cuda.Device(0)
props = cp.cuda.runtime.getDeviceProperties(0)
name  = props.get("name", props.get(b"name"))
free, total = dev.mem_info
print("CuPy        :", cp.__version__ if hasattr(cp, "__version__") else "?", "| CUDA runtime", cp.cuda.runtime.runtimeGetVersion())
print("GPU         :", name.decode() if isinstance(name, bytes) else name)
print("SMs         :", dev.attributes["MultiProcessorCount"], "| compute capability", dev.compute_capability)
print(f"GPU memory  : {total/2**30:.1f} GiB total, {free/2**30:.1f} GiB free")
print("CPU cores   :", os.cpu_count(), "(Colab gives you only a couple, so our NumPy baseline is a *modest* CPU)")

### Two helpers we will use all day

* `timeit(fn, sync=...)`: run a function several times and report the **median** time.
* `sync_gpu()`: wait until the GPU has *really* finished. GPU calls are **asynchronous**: Python continues as soon as the work is *queued*. Without a sync, you time the queueing, not the work (we prove this in Part C).

In [ ]:
def sync_gpu():
    cp.cuda.Stream.null.synchronize()

def timeit(fn, repeat=5, warmup=1, sync=None):
    for _ in range(warmup):
        fn()
    if sync: sync()
    ts = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn()
        if sync: sync()
        ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

results = {}   # we fill this in as we go: backend -> dict(epoch_s, acc)

---
# Part A — Warm-up on the CPU (Weeks 1–4 in 15 minutes)

Before any GPU, let's re-check the habits that will carry over. Everything here runs on the CPU with NumPy.

## A1 · Week 1: *measure first*. How many FLOPs is one training step?

A layer `Z = H @ W + b` with a batch of `B` rows, `n_in` inputs and `n_out` outputs costs `2·B·n_in·n_out` FLOPs
(one multiply + one add per term). A training step is a forward pass plus a backward pass that needs about **two more** matmuls per layer,
so one step ≈ **3 × forward**.

In [ ]:
def step_flops(B, sizes):
    fwd = sum(2 * B * a * b for a, b in zip(sizes[:-1], sizes[1:]))
    return 3 * fwd

B, sizes = 256, [784, 512, 10]
fl = step_flops(B, sizes)
print(f"one training step ≈ {fl/1e6:.0f} MFLOP;  one epoch (60,000 rows = {60000//B} steps) ≈ {fl*(60000//B)/1e9:.0f} GFLOP")

# How fast is this machine's NumPy at its best?  (a big float32 matmul)
a = np.random.rand(2048, 2048).astype(np.float32)
t = timeit(lambda: a @ a)
print(f"NumPy matmul 2048²: {t*1e3:.0f} ms  →  {2*2048**3/t/1e9:.0f} GFLOP/s  (this is the CPU's practical peak for us)")

📝 *Write down:* CPU practical peak = ______ GFLOP/s. A T4 advertises ~8,000 GFLOP/s (FP32). Keep this number: in Part C we measure how much of that we actually get.

## A2 · Week 2: *vectorise*. Loops vs NumPy vs Numba

The same `C = A @ B` three ways. (Pure Python is tiny here, because it is *that* slow.)

In [ ]:
from numba import njit

def matmul_python(A, B):
    n, m, p = len(A), len(B), len(B[0])
    C = [[0.0] * p for _ in range(n)]
    for i in range(n):
        for k in range(m):
            for j in range(p):
                C[i][j] += A[i][k] * B[k][j]
    return C

@njit(fastmath=True)
def matmul_numba(A, B):
    n, m = A.shape; p = B.shape[1]
    C = np.zeros((n, p), dtype=A.dtype)
    for i in range(n):
        for k in range(m):
            for j in range(p):
                C[i, j] += A[i, k] * B[k, j]
    return C

n = 120
A = np.random.rand(n, n).astype(np.float32); Bm = np.random.rand(n, n).astype(np.float32)
t_py = timeit(lambda: matmul_python(A.tolist(), Bm.tolist()), repeat=1, warmup=0)
matmul_numba(A, Bm)                         # first call compiles
t_nb = timeit(lambda: matmul_numba(A, Bm))
t_np = timeit(lambda: A @ Bm)
print(f"n={n}:  pure Python {t_py*1e3:8.1f} ms | Numba loop {t_nb*1e3:7.3f} ms | NumPy @ {t_np*1e3:7.3f} ms")

n = 1024
A = np.random.rand(n, n).astype(np.float32); Bm = np.random.rand(n, n).astype(np.float32)
t_nb = timeit(lambda: matmul_numba(A, Bm), repeat=3)
t_np = timeit(lambda: A @ Bm, repeat=3)
print(f"n={n}: Numba loop {t_nb*1e3:7.1f} ms | NumPy @ {t_np*1e3:7.1f} ms   → tuned BLAS is {t_nb/t_np:.0f}× faster than our own loop")

**Take-away:** the Python loop is hopeless, Numba rescues it, and the vendor-tuned BLAS (`@`) still wins. On the GPU the same story repeats: **cuBLAS** is the tuned matmul, and CuPy's `@` calls it.

## A3 · Week 2: broadcasting and a *stable* softmax

Our network ends with a softmax. The naive formula overflows; subtracting the row maximum fixes it. This is broadcasting: `(B, 10) - (B, 1)`.

In [ ]:
logits = np.array([[1000., 1001., 1002.]], dtype=np.float32)

naive = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)       # exp(1000) = inf
print("naive :", naive)

z = logits - logits.max(axis=1, keepdims=True)                           # (1,3) - (1,1) → broadcast
stable = np.exp(z) / np.exp(z).sum(axis=1, keepdims=True)
print("stable:", stable, "| row sum =", stable.sum())

## A4 · Weeks 3–4: do we need threads or processes for the matmul?

No. `A @ B` already runs on **all your cores inside the BLAS library** and does not hold the GIL (Week 4).
Let's see the threads at work by limiting BLAS to one thread.

In [ ]:
try:
    from threadpoolctl import threadpool_limits
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "threadpoolctl"], check=True)
    from threadpoolctl import threadpool_limits

a = np.random.rand(2048, 2048).astype(np.float32)
for k in sorted({1, os.cpu_count()}):
    with threadpool_limits(limits=k, user_api="blas"):
        t = timeit(lambda: a @ a)
    print(f"BLAS threads = {k}:  {t*1e3:7.1f} ms   ({2*2048**3/t/1e9:5.0f} GFLOP/s)")

Colab's CPU has only 2 cores, so we see ~2×. A GPU has *thousands* of cores. That is the whole motivation for this week.

---
# Part B — The MLP written as matrices (CPU, NumPy)

## B1 · Data = a matrix

MNIST: 60,000 training images of 28×28 pixels, flattened to rows of 784 numbers, with a label 0–9.
We keep everything **`float32`** (a deliberate habit: GPUs are built for 32-bit floats, and NumPy silently produces `float64`).

In [ ]:
def load_mnist():
    path = "mnist.npz"
    if not os.path.exists(path):
        urllib.request.urlretrieve("https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz", path)
    d = np.load(path)
    return (d["x_train"].reshape(-1, 784), d["y_train"].astype(np.int64),
            d["x_test"].reshape(-1, 784),  d["y_test"].astype(np.int64))

def fake_digits(n_train=60000, n_test=10000, seed=0):
    # Offline fallback: 10 noisy "prototype" images. Same shapes as MNIST, easier to learn.
    rng = np.random.default_rng(seed)
    protos = rng.random((10, 784)) * 255
    def make(n):
        y = rng.integers(0, 10, n)
        return np.clip(protos[y] + rng.normal(0, 90, (n, 784)), 0, 255), y.astype(np.int64)
    (Xa, ya), (Xb, yb) = make(n_train), make(n_test)
    return Xa, ya, Xb, yb

try:
    Xtr, ytr, Xte, yte = load_mnist(); source = "MNIST"
except Exception as e:
    print("MNIST download failed (", type(e).__name__, ") → using synthetic digits");
    Xtr, ytr, Xte, yte = fake_digits(); source = "synthetic"

Xtr = Xtr.astype(np.float32) / 255; Xte = Xte.astype(np.float32) / 255
mu, sd = Xtr.mean(axis=0), Xtr.std()          # (784,) per-pixel mean, one global std → broadcasting
Xtr = (Xtr - mu) / sd;  Xte = (Xte - mu) / sd
data = (Xtr, ytr, Xte, yte)
print(source, "| X_train", Xtr.shape, Xtr.dtype, f"({Xtr.nbytes/1e6:.0f} MB)", "| y_train", ytr.shape, "| X_test", Xte.shape)

In [ ]:
fig, axs = plt.subplots(1, 8, figsize=(12, 1.8))
for i, ax in enumerate(axs):
    ax.imshow((Xtr[i] * sd + mu).reshape(28, 28), cmap="gray"); ax.set_title(f"label {ytr[i]}"); ax.axis("off")
plt.show()

## B2 · The model, in four formulas

For a batch `X` (B×784), hidden width `H`, weights `W1` (784×H), `W2` (H×10):

$$H_1=\max(0,\;XW_1+b_1)\qquad L=H_1W_2+b_2\qquad P=\mathrm{softmax}(L)\qquad \mathcal{L}=-\tfrac1B\sum_i\log P_{i,y_i}$$

and, by the chain rule, with `dL = (P − onehot(y)) / B`:

$$\frac{\partial\mathcal L}{\partial W_2}=H_1^{\top}dL\qquad \frac{\partial\mathcal L}{\partial b_2}=\sum_{\text{rows}} dL\qquad dH_1=(dL\,W_2^{\top})\odot[H_1>0]\qquad \frac{\partial\mathcal L}{\partial W_1}=X^{\top}dH_1$$

Count the matmuls: 2 forward, 3 backward (the 5th, `dL·W2ᵀ`, is needed to continue backwards). **Everything else is cheap element-wise work.**

### The one trick that makes the GPU move painless

We write each function so it receives the array module `xp` (either `numpy` or `cupy`).
The code is identical; only the arrays live somewhere else.

In [ ]:
def init_params(sizes, seed=0, dtype=np.float32):
    # He initialisation, always generated on the CPU so every backend starts from identical weights.
    rng = np.random.default_rng(seed)
    params = []
    for n_in, n_out in zip(sizes[:-1], sizes[1:]):
        params.append((rng.standard_normal((n_in, n_out)) * math.sqrt(2.0 / n_in)).astype(dtype))
        params.append(np.zeros(n_out, dtype=dtype))
    return params

def forward(xp, params, X):
    acts, h = [X], X
    n_layers = len(params) // 2
    for l in range(n_layers):
        z = h @ params[2*l] + params[2*l + 1]          # matmul + broadcast bias
        h = xp.maximum(z, 0) if l < n_layers - 1 else z  # ReLU on hidden layers, raw logits at the end
        acts.append(h)
    return acts                                        # acts[-1] are the logits

def softmax_xent(xp, logits, y):
    z = logits - logits.max(axis=1, keepdims=True)
    logp = z - xp.log(xp.exp(z).sum(axis=1, keepdims=True))
    B = logits.shape[0]
    rows = xp.arange(B)
    loss = -logp[rows, y].mean()
    dlogits = xp.exp(logp)                             # softmax probabilities
    dlogits[rows, y] -= 1.0                            # P - onehot(y)
    return loss, dlogits / B

def backward(xp, params, acts, dlogits):
    grads = [None] * len(params)
    d = dlogits
    for l in reversed(range(len(params) // 2)):
        grads[2*l]     = acts[l].T @ d
        grads[2*l + 1] = d.sum(axis=0)
        if l > 0:
            d = (d @ params[2*l].T) * (acts[l] > 0)    # through W, then through the ReLU
    return grads

def loss_and_grads(xp, params, X, y):
    acts = forward(xp, params, X)
    loss, dl = softmax_xent(xp, acts[-1], y)
    return loss, backward(xp, params, acts, dl)

def train_step(xp, params, X, y, lr):
    loss, grads = loss_and_grads(xp, params, X, y)
    for p, g in zip(params, grads):
        p -= lr * g                                    # in-place SGD update
    return loss

## B3 · Correct first, fast second: gradient check

A hand-written backward pass is easy to get wrong, and a wrong model on a fast GPU is just a *fast wrong model*.
We compare our gradients with finite differences, $\frac{L(w+\varepsilon)-L(w-\varepsilon)}{2\varepsilon}$, on a tiny network in `float64`.

In [ ]:
def grad_check(seed=0, eps=1e-6):
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((8, 5)); y = rng.integers(0, 3, 8)
    params = init_params([5, 7, 3], seed, dtype=np.float64)
    params = [p + 0.1 * rng.standard_normal(p.shape) if p.ndim == 1 else p for p in params]   # non-zero biases
    _, grads = loss_and_grads(np, params, X, y)
    worst = 0.0
    for p, g in zip(params, grads):
        flat = p.reshape(-1)                                    # a *view*: editing flat edits p
        for k in rng.choice(flat.size, size=min(6, flat.size), replace=False):
            old = flat[k]
            flat[k] = old + eps; lp, _ = loss_and_grads(np, params, X, y)
            flat[k] = old - eps; lm, _ = loss_and_grads(np, params, X, y)
            flat[k] = old
            num, ana = (lp - lm) / (2 * eps), g.reshape(-1)[k]
            worst = max(worst, abs(num - ana) / max(1e-12, abs(num) + abs(ana)))
    return worst

err = grad_check()
print(f"worst relative error: {err:.2e}", "→ backward() is correct ✔" if err < 1e-5 else "→ BUG!")

## B4 · The training loop (works for NumPy *and* CuPy)

Read the loop carefully, because **we will not change it for the GPU**. Notes for later:

* `Xd, yd = xp.asarray(...)`: with NumPy this is a no-op; with CuPy it is the **one upload** (rule: *in once, out once*).
* the per-epoch shuffle is created on the CPU, uploaded as **one** array, and used to slice batches on the device.
* the running loss `total` stays on the device; we convert it to a Python number **once per epoch**. (`float(loss)` every step would force a GPU→CPU round trip: a *hidden synchronisation*; we measure its cost in Part D.)
* the first few steps are a warm-up on a *copy* of the weights: CuPy compiles kernels and cuBLAS initialises on first use.

In [ ]:
def accuracy(xp, params, X, y, chunk=5000):
    correct = 0
    for i in range(0, X.shape[0], chunk):
        pred = forward(xp, params, X[i:i+chunk])[-1].argmax(axis=1)
        correct += int((pred == y[i:i+chunk]).sum())
    return correct / X.shape[0]

def train(xp, data, hidden=(512,), batch=256, epochs=5, lr=0.1, seed=0,
          resident=True, sync_loss_each_step=False, verbose=True):
    Xtr, ytr, Xte, yte = data
    sync = sync_gpu if xp is cp else (lambda: None)
    params = [xp.asarray(p) for p in init_params([Xtr.shape[1], *hidden, 10], seed)]
    n = Xtr.shape[0]

    t0 = time.perf_counter()
    if resident or xp is np:
        Xd, yd = xp.asarray(Xtr), xp.asarray(ytr)            # ONE upload (a no-op for NumPy)
    Xted, yted = xp.asarray(Xte), xp.asarray(yte)
    sync(); upload_s = time.perf_counter() - t0

    for _ in range(3):                                       # warm-up on throw-away weights
        train_step(xp, [p.copy() for p in params], xp.asarray(Xtr[:batch]), xp.asarray(ytr[:batch]), 0.0)
    sync()

    epoch_s, losses, accs = [], [], []
    for ep in range(epochs):
        perm = np.random.default_rng(seed + ep).permutation(n)
        perm_d = xp.asarray(perm) if (resident or xp is np) else None
        total = xp.zeros((), dtype=xp.float32)
        sync(); t0 = time.perf_counter()
        for i in range(0, n - batch + 1, batch):
            if resident or xp is np:
                idx = perm_d[i:i+batch]
                xb, yb = Xd[idx], yd[idx]
            else:                                            # "streaming": gather on CPU, upload every batch
                idx = perm[i:i+batch]
                xb, yb = xp.asarray(Xtr[idx]), xp.asarray(ytr[idx])
            loss = train_step(xp, params, xb, yb, lr)
            total += loss
            if sync_loss_each_step:
                float(loss)                                  # forces a GPU→CPU round trip every step
        sync(); epoch_s.append(time.perf_counter() - t0)
        losses.append(float(total) / (n // batch))
        accs.append(accuracy(xp, params, Xted, yted))
        if verbose:
            print(f"  epoch {ep+1}/{epochs}: loss {losses[-1]:.4f} | test acc {accs[-1]*100:5.2f}% | {epoch_s[-1]:6.2f} s")
    return dict(params=params, epoch_s=epoch_s, loss=losses, acc=accs, upload_s=upload_s)

### Run it on the CPU (NumPy baseline)

In [ ]:
EPOCHS = 5
print("NumPy on CPU, MLP 784-512-10, batch 256:")
res_np = train(np, data, hidden=(512,), batch=256, epochs=EPOCHS)
results["NumPy (CPU)"] = dict(epoch_s=float(np.median(res_np["epoch_s"])), acc=res_np["acc"][-1])
print(f"→ {results['NumPy (CPU)']['epoch_s']:.2f} s per epoch, final test accuracy {res_np['acc'][-1]*100:.2f}%")

📝 *Write down:* NumPy time per epoch = ______ s, test accuracy = ______ %

## B5 · Week 5 callback: *split, move, wait* with gradients

Week 5's four questions: how to **split**, who does what, what **moves**, who **waits**. Here is data parallelism in six lines.
Split one batch into 4 shards, compute each shard's gradient *independently* (these could be 4 workers!), then **average** the gradients.
The result equals the gradient of the whole batch. This is exactly what DDP (Weeks 10–11) does, with an all-reduce as the "average" step.

In [ ]:
params = init_params([784, 512, 10])
xb, yb = Xtr[:256], ytr[:256]

_, full = loss_and_grads(np, params, xb, yb)
shard_grads = [loss_and_grads(np, params, xs, ys)[1]
               for xs, ys in zip(np.array_split(xb, 4), np.array_split(yb, 4))]        # 4 "workers"
avg = [sum(g[i] for g in shard_grads) / 4 for i in range(len(params))]                  # the "all-reduce"
print("max |full-batch grad − averaged shard grads| =", max(float(np.abs(f - a).max()) for f, a in zip(full, avg)))

## B6 · Week 1 callback: where does one step spend its time? (Amdahl)

Time the five matmuls alone, then the whole step. The fraction `f` of time in matmul tells us how much a *matmul-only* accelerator could help (Amdahl: speedup ≤ 1/(1−f)).

In [ ]:
params = init_params([784, 512, 10]); xb, yb = Xtr[:256], ytr[:256]
W1, W2 = params[0], params[2]
h1 = np.maximum(xb @ W1, 0); d2 = np.random.rand(256, 10).astype(np.float32); d1 = np.random.rand(256, 512).astype(np.float32)

def five_matmuls():
    xb @ W1; h1 @ W2; h1.T @ d2; d2 @ W2.T; xb.T @ d1

t_mm   = timeit(five_matmuls, repeat=20)
t_step = timeit(lambda: train_step(np, params, xb, yb, 0.0), repeat=20)
f = t_mm / t_step
print(f"5 matmuls: {t_mm*1e3:.2f} ms of a {t_step*1e3:.2f} ms step → f = {f:.0%}")
print(f"Amdahl bound if ONLY matmul were infinitely fast: {1/(1-f):.1f}×")
print("→ CuPy moves *everything* to the GPU, so the element-wise 'serial' part also shrinks.")

---
# Part C — Meet the GPU with simple tasks (CuPy)

CuPy mirrors NumPy: `import cupy as cp` and most NumPy code works after swapping the prefix.
The only new ideas are **where the array lives**, **asynchronous execution**, and **moving data**.

## C1 · Hello, GPU

Arrays created with `cp.` live in **GPU memory** (VRAM). Operations on them run as GPU *kernels*.

In [ ]:
x = cp.arange(10, dtype=cp.float32)          # lives in VRAM
y = x * 2 + 1                                # two kernels run on the GPU
print(type(y).__name__, "| shape", y.shape, "| dtype", y.dtype)
print("sum on GPU:", y.sum(), "(still a GPU scalar)")

y_cpu = cp.asnumpy(y)                         # GPU → CPU copy (also written y.get())
print("back on the CPU:", type(y_cpu).__name__, y_cpu)

**Porting rules (keep this list!)**

1. `np.` → `cp.` for creation and math; slicing, `@`, broadcasting, `.sum(axis=...)` are unchanged.
2. Upload **once** with `cp.asarray`, download **once** with `cp.asnumpy` (or `.get()`).
3. Stay in `float32`. NumPy's default `float64` is *slow* on most GPUs (Part C5).
4. GPU calls return **immediately**. Synchronise before you read a clock.
5. Anything that returns a Python number (`float(x)`, `.item()`, `print(x)`, `if x > 0`) **waits for the GPU**.
6. Write functions as `f(xp, ...)` to run on both. That is what we did in Part B.

## C2 · Simple task 1: element-wise ReLU on 10 million numbers (and a timing trap)

In [ ]:
n = 10_000_000
x_cpu = np.random.standard_normal(n).astype(np.float32)
x_gpu = cp.asarray(x_cpu)
cp.maximum(x_gpu, 0); sync_gpu()                     # warm-up (first call compiles the kernel)

# ❌ WRONG: time only the *launch*
t0 = time.perf_counter(); y = cp.maximum(x_gpu, 0); t_wrong = time.perf_counter() - t0
sync_gpu()

# ✅ RIGHT: wait for the GPU before reading the clock
t_gpu = timeit(lambda: cp.maximum(x_gpu, 0), sync=sync_gpu, repeat=20)
t_cpu = timeit(lambda: np.maximum(x_cpu, 0), repeat=20)

print(f"NumPy                       : {t_cpu*1e3:8.3f} ms")
print(f"CuPy, no sync (WRONG)       : {t_wrong*1e3:8.3f} ms   ← just the time to queue the kernel!")
print(f"CuPy, with sync (correct)   : {t_gpu*1e3:8.3f} ms   → {t_cpu/t_gpu:.1f}× faster than NumPy")
gbs = 2 * 4 * n / t_gpu / 1e9
print(f"memory traffic {gbs:.0f} GB/s (T4 peak ≈ 320 GB/s): ReLU does almost no math, so it is *bandwidth*-bound")

📝 *Write down:* the wrong measurement says ______ ms, the right one ______ ms. Why is the wrong number so small?

**Kernel fusion preview.** `1 / (1 + exp(-x))` (sigmoid) launches *four* kernels and creates three temporary arrays in VRAM.
`cp.fuse` merges them into one kernel: each element is read once and written once. (In Week 8 you will write such kernels yourself.)

In [ ]:
def sigmoid(x):
    return 1 / (1 + cp.exp(-x))

@cp.fuse()
def sigmoid_fused(x):
    return 1 / (1 + cp.exp(-x))

sigmoid(x_gpu); sigmoid_fused(x_gpu); sync_gpu()
t1 = timeit(lambda: sigmoid(x_gpu), sync=sync_gpu, repeat=20)
t2 = timeit(lambda: sigmoid_fused(x_gpu), sync=sync_gpu, repeat=20)
t0 = timeit(lambda: 1 / (1 + np.exp(-x_cpu)), repeat=5)
print(f"NumPy {t0*1e3:7.2f} ms | CuPy 4 kernels {t1*1e3:6.3f} ms | CuPy fused {t2*1e3:6.3f} ms")

## C3 · Simple task 2: matrix multiplication, where the GPU really shines

Matmul does `2n³` FLOPs on `3n²` numbers: lots of arithmetic per byte. Three timings per size:

* **NumPy** (CPU, multi-threaded BLAS)
* **CuPy compute only** (data already on the GPU)
* **CuPy end-to-end** (upload both matrices + matmul + download the result)

📝 *Predict first:* at what `n` will the GPU win on compute-only? And end-to-end? ______

In [ ]:
sizes = [128, 256, 512, 1024, 2048, 4096]
rows = []
for n in sizes:
    A = np.random.rand(n, n).astype(np.float32); Bm = np.random.rand(n, n).astype(np.float32)
    Ag, Bg = cp.asarray(A), cp.asarray(Bm)
    t_cpu = timeit(lambda: A @ Bm, repeat=3 if n >= 2048 else 7)
    t_gpu = timeit(lambda: Ag @ Bg, sync=sync_gpu, repeat=20)
    t_e2e = timeit(lambda: cp.asnumpy(cp.asarray(A) @ cp.asarray(Bm)), repeat=5)
    rows.append((n, t_cpu, t_gpu, t_e2e))

print(f"{'n':>5} | {'NumPy ms':>9} | {'GPU ms':>8} | {'GPU end-to-end ms':>17} | {'speedup':>8} | {'GPU GFLOP/s':>11}")
for n, tc, tg, te in rows:
    print(f"{n:>5} | {tc*1e3:9.2f} | {tg*1e3:8.3f} | {te*1e3:17.2f} | {tc/tg:7.1f}× | {2*n**3/tg/1e9:11.0f}")

In [ ]:
ns = [r[0] for r in rows]
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
axs[0].loglog(ns, [r[1] for r in rows], "o-", label="NumPy (CPU)")
axs[0].loglog(ns, [r[2] for r in rows], "o-", label="CuPy compute only")
axs[0].loglog(ns, [r[3] for r in rows], "o--", label="CuPy end-to-end")
axs[0].set_xlabel("n (matrix is n×n)"); axs[0].set_ylabel("seconds"); axs[0].legend(); axs[0].set_title("matmul time")
axs[1].semilogx(ns, [r[1]/r[2] for r in rows], "o-", label="compute only")
axs[1].semilogx(ns, [r[1]/r[3] for r in rows], "o--", label="end-to-end")
axs[1].axhline(1, color="gray", lw=1); axs[1].set_xlabel("n"); axs[1].set_ylabel("speedup over NumPy"); axs[1].legend(); axs[1].set_title("GPU speedup (above 1 = GPU wins)")
plt.tight_layout(); plt.show()

📝 *Questions*

1. Compute-only: the GPU first wins at n = ______. End-to-end: n = ______.
2. Matmul compute grows like n³ but transfer grows like n². Why does the GPU *inevitably* win as n grows? (Think Amdahl.)
3. The "GPU GFLOP/s" column against the T4's ~8,000 peak: what percentage do we reach at n = 4096? And the CPU number from A1?

## C4 · The transfer trap: what does one trip over PCIe cost?

In [ ]:
print(f"{'size':>10} | {'upload':>10} | {'bandwidth':>10}")
for mb in [0.004, 0.4, 4, 40, 400]:
    n = int(mb * 1e6 / 4)
    h = np.random.rand(n).astype(np.float32)
    t = timeit(lambda: cp.asarray(h), sync=sync_gpu, repeat=10)
    print(f"{mb:8.3f} MB | {t*1e6:7.0f} µs | {n*4/t/1e9:6.1f} GB/s")
print("\nA tiny array pays a fixed latency (≈10–50 µs) and sees almost no bandwidth: don't ping-pong small arrays.")

**Rule of the week:** *upload once → compute a lot → download once.* Now the same idea for training data: MNIST is 188 MB, which uploads in tens of milliseconds. We do that **once**, not once per batch (we'll measure the difference in Part D).

## C5 · Why float32? One more reason

Consumer and Colab GPUs have far fewer FP64 units than FP32 units (T4: 1/32). NumPy's default dtype is `float64`: forget `.astype(np.float32)` and you silently run on the slow units.

In [ ]:
n = 4096
for dt in (cp.float32, cp.float64):
    a = cp.random.standard_normal((n, n), dtype=dt)
    t = timeit(lambda: a @ a, sync=sync_gpu, repeat=5)
    print(f"{cp.dtype(dt).name:>8}: {t*1e3:8.1f} ms   {2*n**3/t/1e12:5.2f} TFLOP/s")

---
# Part D — Train the *same* MLP on the GPU

Now the payoff: call `train()` with `xp = cp`. **Not one line of the model or the loop changes.**

In [ ]:
print("CuPy on GPU, MLP 784-512-10, batch 256:")
res_cp = train(cp, data, hidden=(512,), batch=256, epochs=EPOCHS)
results["CuPy (GPU)"] = dict(epoch_s=float(np.median(res_cp["epoch_s"])), acc=res_cp["acc"][-1])

sp = results["NumPy (CPU)"]["epoch_s"] / results["CuPy (GPU)"]["epoch_s"]
print(f"→ {results['CuPy (GPU)']['epoch_s']:.3f} s per epoch (NumPy: {results['NumPy (CPU)']['epoch_s']:.2f} s) → {sp:.1f}× faster; "
      f"one-time upload {res_cp['upload_s']*1e3:.0f} ms; test accuracy {res_cp['acc'][-1]*100:.2f}%")

diff = max(float(np.abs(cp.asnumpy(a) - b).max()) for a, b in zip(res_cp["params"], res_np["params"]))
print(f"max |weight difference| GPU vs CPU after {EPOCHS} epochs: {diff:.2e}  (same maths, different order of float additions)")

📝 *Write down:* GPU time per epoch = ______ s, speedup = ______×, test accuracy = ______ %

*Why isn't the speedup closer to what the peak numbers promise?* A T4 can do several TFLOP/s but our step is only 0.6 GFLOP, i.e. roughly **0.1–0.2 ms of real math**.
Each step launches dozens of small kernels with ~10 µs of launch overhead each, so a model this small cannot keep the GPU busy. The sweep below tests this explanation.

## D1 · When does the GPU win? A sweep over model size and batch size

We time a single training step (no data loading) for a grid of hidden widths `H` and batch sizes `B`.

In [ ]:
Hs = [64, 256, 1024, 4096]
Bs = [32, 128, 512, 2048]
speed = np.zeros((len(Bs), len(Hs)))
t_cpu_grid = np.zeros_like(speed); t_gpu_grid = np.zeros_like(speed)
for i, B in enumerate(Bs):
    for j, H in enumerate(Hs):
        p0 = init_params([784, H, 10])
        xb = np.random.standard_normal((B, 784)).astype(np.float32); yb = np.random.randint(0, 10, B)
        pc = [p.copy() for p in p0]
        t_c = timeit(lambda: train_step(np, pc, xb, yb, 0.0), repeat=5)
        pg = [cp.asarray(p) for p in p0]; xg, yg = cp.asarray(xb), cp.asarray(yb)
        t_g = timeit(lambda: train_step(cp, pg, xg, yg, 0.0), sync=sync_gpu, repeat=20)
        t_cpu_grid[i, j], t_gpu_grid[i, j], speed[i, j] = t_c, t_g, t_c / t_g

fig, ax = plt.subplots(figsize=(6.4, 4.2))
im = ax.imshow(np.log10(speed), cmap="RdYlGn", vmin=-1, vmax=2)
ax.set_xticks(range(len(Hs)), [f"H={h}" for h in Hs]); ax.set_yticks(range(len(Bs)), [f"B={b}" for b in Bs])
for i in range(len(Bs)):
    for j in range(len(Hs)):
        ax.text(j, i, f"{speed[i, j]:.1f}×", ha="center", va="center", fontsize=11)
ax.set_title("GPU speedup over NumPy, one training step\n(green: GPU wins, red: CPU wins)"); plt.tight_layout(); plt.show()

📝 *Questions*

1. In which corner of the grid does the CPU win or tie? Explain with *kernel launch overhead*.
2. Which corner has the biggest speedup? What does a larger batch or width do to the work per kernel?
3. Practical advice for a student training a tiny model: is a GPU always worth it?

## D2 · Three ways to throw the speedup away

Same model, same GPU, one epoch each. Only the *style* changes.

In [ ]:
cfgs = [("baseline: data on GPU, loss synced once/epoch", dict()),
        ("hidden sync: float(loss) every step",           dict(sync_loss_each_step=True)),
        ("transfer trap: upload every batch from CPU",    dict(resident=False))]
for label, kw in cfgs:
    r = train(cp, data, hidden=(512,), batch=256, epochs=1, verbose=False, **kw)
    print(f"{label:<50}: {r['epoch_s'][0]:6.3f} s/epoch")
r = train(np, data, hidden=(512,), batch=256, epochs=1, verbose=False)
print(f"{'(reference) NumPy on CPU':<50}: {r['epoch_s'][0]:6.3f} s/epoch")

**Take-away.** The GPU gives its speedup only if the loop stays *on* the device: no per-step `float(loss)`, no per-batch uploads.
(In Week 9 you will see how `DataLoader`, `pin_memory` and `non_blocking` hide the unavoidable transfers.)

## D3 · GPU memory: the other limit

CuPy keeps freed blocks in a **memory pool** to avoid slow `cudaMalloc` calls. `nvidia-smi` therefore shows more than your live arrays.

In [ ]:
pool = cp.get_default_memory_pool()
print(f"in use by live arrays: {pool.used_bytes()/2**20:7.1f} MiB | held by the pool: {pool.total_bytes()/2**20:7.1f} MiB")
pool.free_all_blocks()
print(f"after free_all_blocks : {pool.used_bytes()/2**20:7.1f} MiB | held by the pool: {pool.total_bytes()/2**20:7.1f} MiB")
print("Memory per step ≈ weights + activations (B×H floats per layer) + gradients. Big batches or models hit 'out of memory' first, not 'too slow'.")

---
# Part E — The same model in PyTorch

CuPy is "NumPy on the GPU": it has no autograd, so we wrote the backward pass by hand. PyTorch
gives the same GPU arrays (tensors) **plus automatic differentiation**. From Week 9 on we use PyTorch for real training.

## E1 · Does autograd compute *our* gradients?

We build the same network, copy our initial weights in, and compare the gradients on one batch.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available(), "| CPU threads:", torch.get_num_threads())

def make_torch_model(hidden, seed=0):
    sizes = [784, *hidden, 10]
    layers = []
    for a, b in zip(sizes[:-1], sizes[1:]):
        layers += [nn.Linear(a, b), nn.ReLU()]
    model = nn.Sequential(*layers[:-1])                      # no ReLU after the last layer
    params = init_params(sizes, seed)                        # identical start to the NumPy/CuPy runs
    with torch.no_grad():
        for lin, (W, b) in zip([m for m in model if isinstance(m, nn.Linear)], zip(params[0::2], params[1::2])):
            lin.weight.copy_(torch.from_numpy(W.T.copy())); lin.bias.copy_(torch.from_numpy(b))
    return model

model = make_torch_model((512,))
xb, yb = Xtr[:256], ytr[:256]
F.cross_entropy(model(torch.from_numpy(xb)), torch.from_numpy(yb)).backward()

_, ours = loss_and_grads(np, init_params([784, 512, 10]), xb, yb)
lins = [m for m in model if isinstance(m, nn.Linear)]
worst = max(float(np.abs(lins[0].weight.grad.numpy().T - ours[0]).max()), float(np.abs(lins[1].weight.grad.numpy().T - ours[2]).max()))
print(f"max |autograd − hand-written gradient| = {worst:.2e}  → autograd computes exactly the matrices we derived ✔")

## E2 · Train it on CPU and GPU with PyTorch

In [ ]:
def train_torch(device, data, hidden=(512,), batch=256, epochs=5, lr=0.1, seed=0, verbose=True):
    Xtr, ytr, Xte, yte = data
    sync = torch.cuda.synchronize if device.type == "cuda" else (lambda: None)
    model = make_torch_model(hidden, seed).to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    Xd, yd = torch.from_numpy(Xtr).to(device), torch.from_numpy(ytr).to(device)       # ONE upload
    Xted, yted = torch.from_numpy(Xte).to(device), torch.from_numpy(yte).to(device)
    n = Xd.shape[0]

    for _ in range(3):                                      # warm-up (cuBLAS init, allocator)
        F.cross_entropy(model(Xd[:batch]), yd[:batch]).backward()
    model.zero_grad(set_to_none=True); sync()

    epoch_s, accs = [], []
    for ep in range(epochs):
        perm = torch.from_numpy(np.random.default_rng(seed + ep).permutation(n)).to(device)
        total = torch.zeros((), device=device)
        sync(); t0 = time.perf_counter()
        for i in range(0, n - batch + 1, batch):
            idx = perm[i:i+batch]
            loss = F.cross_entropy(model(Xd[idx]), yd[idx])
            opt.zero_grad(set_to_none=True)
            loss.backward()                                  # autograd = our backward()
            opt.step()                                       # SGD update
            total += loss.detach()
        sync(); epoch_s.append(time.perf_counter() - t0)
        with torch.no_grad():
            accs.append(float((model(Xted).argmax(1) == yted).float().mean()))
        if verbose:
            print(f"  epoch {ep+1}/{epochs}: loss {float(total)/(n//batch):.4f} | test acc {accs[-1]*100:5.2f}% | {epoch_s[-1]:6.2f} s")
    return dict(epoch_s=epoch_s, acc=accs)

print("PyTorch on CPU:")
r_tc = train_torch(torch.device("cpu"), data, epochs=EPOCHS)
results["PyTorch (CPU)"] = dict(epoch_s=float(np.median(r_tc["epoch_s"])), acc=r_tc["acc"][-1])
print("PyTorch on GPU:")
r_tg = train_torch(torch.device("cuda"), data, epochs=EPOCHS)
results["PyTorch (GPU)"] = dict(epoch_s=float(np.median(r_tg["epoch_s"])), acc=r_tg["acc"][-1])

## E3 · Mixed-precision preview (Week 12): Tensor Cores

The T4 has Tensor Cores that multiply **float16** matrices much faster than the normal FP32 cores. One matmul, two precisions:

In [ ]:
n = 4096
a32 = torch.randn(n, n, device="cuda"); a16 = a32.half()
for name, a in [("float32", a32), ("float16 (Tensor Cores)", a16)]:
    t = timeit(lambda: a @ a, sync=torch.cuda.synchronize, repeat=10)
    print(f"{name:>22}: {t*1e3:7.2f} ms  → {2*n**3/t/1e12:5.1f} TFLOP/s")
print("Lower precision = fewer bytes to move AND faster math. The price is accuracy care (loss scaling): Week 12.")

---
# Part F — Wrap-up

In [ ]:
print(f"{'backend':<16} | {'s / epoch':>10} | {'vs NumPy':>9} | {'test acc':>8}")
base = results["NumPy (CPU)"]["epoch_s"]
for k, v in results.items():
    print(f"{k:<16} | {v['epoch_s']:10.3f} | {base/v['epoch_s']:8.1f}× | {v['acc']*100:7.2f}%")

fig, ax = plt.subplots(figsize=(6.4, 3.2))
ks = list(results); vals = [results[k]["epoch_s"] for k in ks]
bars = ax.barh(ks, vals, color=["#6b7280", "#2f6fb0", "#e08a1e", "#16a34a"][:len(ks)])
ax.set_xscale("log"); ax.set_xlabel("seconds per epoch (log scale, lower is better)"); ax.invert_yaxis()
for b, v in zip(bars, vals): ax.text(v, b.get_y() + b.get_height()/2, f" {v:.2f}s", va="center")
plt.tight_layout(); plt.show()

## 📝 Your wrap-up (submit with your results)

1. Your completed tables: A1 CPU GFLOP/s, B4 NumPy epoch time, C3 crossover size, D epoch time and speedup.
2. One sentence: *"On a T4, training this MLP on the GPU is worthwhile when … because …"* (use numbers from the sweep in D1).
3. The four course questions for the GPU: what is its answer to **split**, **assign**, **move**, **wait**? (Hints: one thread per element · the hardware block scheduler · `asarray`/`asnumpy` over PCIe · asynchronous launches + `synchronize`.)
4. PyTorch vs CuPy on this model gave similar epoch times. Why? What would PyTorch do better on a bigger model? (Hint: autograd bookkeeping vs launch overhead; mixed precision; `torch.compile`.)

## 🔧 Exercises (pick two)

* **E-1 Deeper network.** Train `hidden=(512, 256)`. The code already supports it. Is the GPU speedup larger or smaller? Why?
* **E-2 Batch size.** Train with `batch=64` and `batch=2048` on the GPU. Plot time per epoch and final accuracy. What is the trade-off?
* **E-3 Momentum.** Add SGD with momentum to `train_step` (keep one velocity array per parameter, using in-place updates). Check the gradient check still passes, then check that it trains faster per epoch.
* **E-4 Float16 in CuPy.** Cast the weights and data to `cp.float16`. Does the loss still go down? Compare time per epoch. (Hint: you may need a smaller learning rate.)
* **E-5 Honest benchmark.** Compare against a CPU with more cores (your laptop?). How does the speedup change when NumPy gets 8 cores instead of 2?

---
*Next week: midterm (Weeks 1–6). Week 8: write your own CUDA kernels with Numba.*